In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata
import os
import seaborn as sb
import matplotlib
from harmony import harmonize

In [2]:
import rpy2.rinterface_lib.callbacks
import logging

from rpy2.robjects import pandas2ri
import anndata2ri

# Ignore R warning messages
#Note: this can be commented out to get more verbose R output
rpy2.rinterface_lib.callbacks.logger.setLevel(logging.ERROR)

# Automatically convert rpy2 outputs to pandas dataframes
pandas2ri.activate()
anndata2ri.activate()
%load_ext rpy2.ipython

/tmp/ipykernel_3573802/2253401465.py:13: DeprecationWarning: The global conversion available with activate() is deprecated and will be removed in the next major release. Use a local converter.
  anndata2ri.activate()


# T_cells

In [3]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/T_cells/T_cells-reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/T_cells/T_cells-reload-harm-preannot.csv",row.names = FALSE, )


    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast    

Loading required package: SeuratObject
Loading required package: sp

Attaching package: ‘SeuratObject’

The following objects are masked from ‘package:base’:

    intersect, t


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



# ECs

In [4]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/ECs/EC-reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/ECs/EC-reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...


# Macrophages

In [5]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/Macrophages/Macrophage-reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/Macrophages/Macrophage-reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...


# DCs

In [6]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/DCs/DC-reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/DCs/DC-reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...


# NK

In [7]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/NK/NK-reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/NK/NK-reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...


# B cells

In [8]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/B_cells/B_cell-reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/B_cells/B_cell-reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...


# Monocytes

In [9]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/Monocytes/Monocyte-reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/Monocytes/Monocyte-reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...


# Neutrophil

In [10]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/Neutrophil/Neutrophil-reload-harm-preannot.rds")
level1_marker <- list(
    'B cell' = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/Neutrophil/Neutrophil-reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...


# Fibro/fibromyo/smc

In [12]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/Fibro_smc/fibrossmc_adata_reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/Fibro_smc/fibrossmc_adata_reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...


# mast cell

In [11]:
%%R 
library(Seurat)
library(openxlsx)
library(Matrix)
library(stringr)
library(dplyr)
adata <- readRDS("./output_260420/query_human_260420/Mast_cells/mast_adata_reload-harm-preannot.rds")
level1_marker <- list(
    "B cell" = c('CD79A', 'CD79B', 'MS4A1', 'IGKC', 'CD22', 'FCER2'),
    'T cell'= c('CD2', 'TRAC', 'CD69', 'CD3D', 'CD3E', 'CD4', 'CD8A', 'CD8B', 'EOMES', 'LAG3'),
    'Natural killer cell'= c('NKG7', 'XCL1', 'CTSW', 'XCL2', 'CD160', 'FCGR3A', 'PRF1', 'GNLY'),

    'Dendritic cell'= c('CLEC10A', 'FCER1A', 'CD1C', 'HLA-DRA', 'HLA-DRB1'),
    'Macrophage'= c('C1QA', 'C1QB', 'C1QC', 'CD74', 'CXCL8', 'AIF1', 'CD68', 'ITGAM', 'CSF1R', 'HLA-DRA', 'LGALS3','CD163'),##CD163 from cc,del CD14
    'Monocyte'= c('FCN1', 'S100A8', 'S100A9', 'S100A12', 'VCAN', 'CD52', 'LYZ', 'CTSS','CD14'),## CD14 from CC
    'Mast cell'= c('TPSAB1', 'TPSB2',  'HDC', 'CMA1') ,
    'Erythrocyte/Erythroid'= c('HBB', 'HBA1', 'HBA2', 'ALAS2', 'AHSP', 'SLC4A1', 'GYPA', 'KLF1', 'TMCC2'), 
    'Neutrophil' = c('NAMPT','IFITM2','G0S2','CXCL8','NEAT1','SRGN','AQP9','SOD2','FCGR3B','IVNS1ABP'),
    # 'Basophil'= c('IL3RA', 'FCER1A', 'MS4A2', 'HDC', 'GATA1', 'PRSS33', 'ENPP3'),
    'Basophil' =  c('TPSB2', 'CPA3', 'SLC24A3', 'FER', 'RP11-779O18.3', 'KIT', 'HPGDS', 'SYTL3', 'MAML3', 'ELL2', 'RP11-139E24.1', 'CCDC200', 'AKAP13', 'AREG', 'RHOH', 'LRMDA', 'ARID1B', 'IRAK3', 'TEX14', 'HPGD', 'ERCC1', 'CTNNBL1', 'LINC00486_ENSG00000230876', 'ZBTB20'),

    'Endothelial cell'= c('PECAM1', 'VWF', 'FABP4', 'CLDN5', 'IFI27', 'ECSCR', 'DYSF', 'CD34', 'COL4A1', 'COL4A2', 'SPARCL1', 'PLVAP', 'MPZL2', 'SULF1', 'EDN1'),

    'Fibroblast'= c('LUM', 'DCN', 'COL1A1', 'COL1A2', 'FBLN1', 'THY1'), 
    'Smooth muscle cell'= c('ACTA2', 'MYH11', 'MYL9', 'TPM2', 'CALD1', 'TAGLN', 'TNFRSF11B', 'LUM', 'APOE', 'APOC1', 'AGT', 'NOTCH3', 'PDGFRB', 'MFAP4'), 
    
    # 'Pericyte'= c('RGS5', 'PDGFRB', 'CSPG4', 'KCNJ8', 'ABCC9', 'NOTCH3', 'MCAM', 'DES', 'MYLK', 'ACTN1')
    'Pericyte'= c('TAGLN', 'LPP', 'CALD1', 'TPM2', 'MYL9', 'ACTA2', 'MAP1B', 'PRKG1', 'IGFBP5', 'SYNPO2', 'EPS8', 'TIMP3', 'LMOD1', 'C11orf96', 'INPP4B', 'NOTCH3', 'EBF1', 'STEAP4', 'MT-RNR1', 'CRISPLD2', 'SOX5', 'PPP1R14A', 'FILIP1L', 'LHFPL6', 'PTPRG')
)
str(level1_marker)

sc <- AddModuleScore(object = adata,features = level1_marker,ctrl = 100,name = "ModuleScore_",assay = "RNA",slot = "counts")
module_score_cols <- grep("ModuleScore_", colnames(sc@meta.data), value = TRUE)
#按聚类分组，计算每个模块的平均得分
cluster_module_score_table <- sc@meta.data %>%group_by(leiden) %>%summarise(across(all_of(module_score_cols), mean), .groups = "drop") %>%
  rename(Cluster = leiden)  
cluster_top_module <- apply(cluster_module_score_table[, -1], 1, function(x) colnames(cluster_module_score_table[, -1])[which.max(x)])
cluster_top_celltype <- gsub("ModuleScore_", "", cluster_top_module)
cluster_module_score_table$Top_Matched_CellType <- cluster_top_celltype
write.csv(cluster_module_score_table,file = "./output_260420/query_human_260420/Mast_cells/mast_adata_reload-harm-preannot.csv",row.names = FALSE, )

List of 14
 $ B cell               : chr [1:6] "CD79A" "CD79B" "MS4A1" "IGKC" ...
 $ T cell               : chr [1:10] "CD2" "TRAC" "CD69" "CD3D" ...
 $ Natural killer cell  : chr [1:8] "NKG7" "XCL1" "CTSW" "XCL2" ...
 $ Dendritic cell       : chr [1:5] "CLEC10A" "FCER1A" "CD1C" "HLA-DRA" ...
 $ Macrophage           : chr [1:12] "C1QA" "C1QB" "C1QC" "CD74" ...
 $ Monocyte             : chr [1:9] "FCN1" "S100A8" "S100A9" "S100A12" ...
 $ Mast cell            : chr [1:4] "TPSAB1" "TPSB2" "HDC" "CMA1"
 $ Erythrocyte/Erythroid: chr [1:9] "HBB" "HBA1" "HBA2" "ALAS2" ...
 $ Neutrophil           : chr [1:10] "NAMPT" "IFITM2" "G0S2" "CXCL8" ...
 $ Basophil             : chr [1:24] "TPSB2" "CPA3" "SLC24A3" "FER" ...
 $ Endothelial cell     : chr [1:15] "PECAM1" "VWF" "FABP4" "CLDN5" ...
 $ Fibroblast           : chr [1:6] "LUM" "DCN" "COL1A1" "COL1A2" ...
 $ Smooth muscle cell   : chr [1:14] "ACTA2" "MYH11" "MYL9" "TPM2" ...
 $ Pericyte             : chr [1:25] "TAGLN" "LPP" "CALD1" "TPM2" ...
